# モーメント編集データの UMAP

- real CIFAR-10（train 全50,000件）に UMAP を fit し，**その reducer をそのまま使って** gram_k1〜k6 のデータを `transform` する
- 同じ射影の上で k を変えたときに埋め込みがどう変化するかを可視化する
- UMAP の使い方は `umap_tutorial.ipynb`（tutorials からのコピー）を参照
- `BACKEND` で GPU（cuML）/ CPU（umap-learn）を切り替える．GPU サーバー（gpu1/gpu2）では `uv sync --extra gpu` で cuML が入る

## データの対応関係
- モーメント編集データ（`~/server/dsb_llc/input/moment_data/deletion/gram_k*/*.npz`）は
  - `pixel_values`: (50000, 32, 32, 3) float32, [0, 1]
  - `original_label`, `target_label`, `sample_index`: (50000,)
- `sample_index` は **TFDS の cifar10 train の並び順** でのインデックス（生成スクリプトが `tfds` で読み込んでいるため）
- TFDS の並び順はバイナリ版（`cifar-10-batches-bin`）の並び順と異なるので，TFDS の tfrecord に入っている `id`（`train_XXXXX`，バイナリ版での位置）を読んで並べ替える
  - TensorFlow を入れずに済むように tfrecord から `id` だけを正規表現で取り出す

In [ ]:
import glob
import os
import re
import struct

import time

import matplotlib.pyplot as plt
import numpy as np

SERVER_DIR = os.path.expanduser("~/server/dsb_llc/input/moment_data/deletion")
TFDS_DIR = os.path.expanduser("~/tensorflow_datasets")
TFRECORD = os.path.join(TFDS_DIR, "cifar10", "3.0.2", "cifar10-train.tfrecord-00000-of-00001")

KS = [1, 2, 3, 4, 5, 6]
N_SAMPLES = None  # None なら全件（50,000件）．動作確認用に減らすときは 10000 など
SEED = 42

# "auto": cuML（GPU）が import できればそれを使い，できなければ umap-learn（CPU）
# GPU 版は Linux サーバーで `uv sync --extra gpu` したときだけ入る
BACKEND = "auto"
if BACKEND in ("auto", "gpu"):
    try:
        from cuml.manifold import UMAP
        BACKEND = "gpu"
    except ImportError:
        if BACKEND == "gpu":
            raise
        BACKEND = "cpu"
if BACKEND == "cpu":
    from umap import UMAP
print("backend:", BACKEND)

cifar_classes = ["airplane", "automobile", "bird", "cat", "deer",
                 "dog", "frog", "horse", "ship", "truck"]

In [ ]:
def load_tfds_ids(path=TFRECORD):
    """TFDS の tfrecord から各レコードの id（バイナリ版での位置）を順に読む"""
    ids = []
    with open(path, "rb") as f:
        while True:
            header = f.read(12)  # length (uint64) + crc (uint32)
            if not header:
                break
            n = struct.unpack("<Q", header[:8])[0]
            record = f.read(n)
            f.read(4)  # data crc
            ids.append(int(re.search(rb"train_(\d+)", record).group(1)))
    return np.array(ids)


def load_real_cifar10_tfds_order():
    """real CIFAR-10 train を TFDS の並び順で読み込む（画素は [0, 1]，(N, 32, 32, 3)）"""
    batch_dir = glob.glob(os.path.join(TFDS_DIR, "downloads", "extracted", "*", "cifar-10-batches-bin"))[0]
    rows = np.concatenate([
        np.fromfile(os.path.join(batch_dir, f"data_batch_{i}.bin"), dtype=np.uint8).reshape(-1, 3073)
        for i in range(1, 6)
    ])
    rows = rows[load_tfds_ids()]
    X = rows[:, 1:].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    y = rows[:, 0].astype(int)
    return X, y


def load_moment_data(k):
    """gram_k のモーメント編集データを読み込む"""
    path = glob.glob(os.path.join(SERVER_DIR, f"gram_k{k}", "*.npz"))[0]
    with np.load(path) as d:
        return {key: d[key] for key in ["pixel_values", "original_label", "sample_index"]}

## real データの読み込みとサブサンプル

In [ ]:
X_real_all, y_real_all = load_real_cifar10_tfds_order()
print("real:", X_real_all.shape, y_real_all.shape, np.bincount(y_real_all))

if N_SAMPLES is None:
    N_SAMPLES = len(X_real_all)
    idx = np.arange(N_SAMPLES)
else:
    rng = np.random.default_rng(SEED)
    idx = np.sort(rng.choice(len(X_real_all), size=N_SAMPLES, replace=False))  # TFDS 順のインデックス
X_real = X_real_all[idx].reshape(N_SAMPLES, -1)
y_real = y_real_all[idx]
print("subsample:", X_real.shape)

## gram_k1〜k6 の読み込み
- real のサブサンプルと **同じ画像** を `sample_index` で取り出し，行の順番も real に揃える
- ラベルが一致するか，画像がどれくらい変わったか（real との差の RMS）を確認する

In [ ]:
# サブサンプルのときは抽出結果をローカルにキャッシュする（Mac で VPN 越しに読むのを2回目以降省く）
# 全件のときはキャッシュしない（サーバー上ではデータがローカルにあり，キャッシュは 3.7GB になるため）
USE_CACHE = N_SAMPLES < 50000
CACHE_PATH = f"../data/gram_k_subsample_n{N_SAMPLES}_seed{SEED}.npz"

X_moment = {}
if USE_CACHE and os.path.exists(CACHE_PATH):
    with np.load(CACHE_PATH) as c:
        assert (c["idx"] == idx).all()
        X_moment = {k: c[f"k{k}"] for k in KS}
for k in KS:
    if k in X_moment:
        print(f"k={k}: loaded from cache, RMS diff from real={np.sqrt(np.mean((X_moment[k] - X_real) ** 2)):.4f}")
        continue
    d = load_moment_data(k)
    pos = np.empty(len(d["sample_index"]), dtype=int)
    pos[d["sample_index"]] = np.arange(len(d["sample_index"]))  # TFDS 順 → npz 内の行
    rows = pos[idx]
    assert (d["original_label"][rows] == y_real).all(), f"k={k}: label mismatch"
    X_moment[k] = d["pixel_values"][rows].reshape(N_SAMPLES, -1)
    rms = np.sqrt(np.mean((X_moment[k] - X_real) ** 2))
    print(f"k={k}: shape={X_moment[k].shape}, RMS diff from real={rms:.4f}")
    del d

if USE_CACHE and not os.path.exists(CACHE_PATH):
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    np.savez(CACHE_PATH, idx=idx, **{f"k{k}": X_moment[k] for k in KS})

In [ ]:
# 同じ画像が k でどう変わるかを並べて確認
n_show = 6
fig, axes = plt.subplots(n_show, len(KS) + 1, figsize=(1.6 * (len(KS) + 1), 1.6 * n_show))
for i in range(n_show):
    imgs = [X_real[i]] + [X_moment[k][i] for k in KS]
    titles = ["real"] + [f"k={k}" for k in KS]
    for j, (img, t) in enumerate(zip(imgs, titles)):
        ax = axes[i, j]
        ax.imshow(np.clip(img.reshape(32, 32, 3), 0, 1))
        ax.axis("off")
        if i == 0:
            ax.set_title(t)
    axes[i, 0].text(-4, 16, cifar_classes[y_real[i]], ha="right", va="center", fontsize=8)
plt.tight_layout()

## real に UMAP を fit し，同じ reducer で k=1〜6 を transform

In [ ]:
reducer = UMAP(n_components=2, random_state=SEED)
t0 = time.time()
Z_real = np.asarray(reducer.fit_transform(X_real))
print(f"fit (real): {time.time() - t0:.1f}s")

Z_moment = {}
for k in KS:
    t0 = time.time()
    Z_moment[k] = np.asarray(reducer.transform(X_moment[k]))
    print(f"transform (k={k}): {time.time() - t0:.1f}s")

# 埋め込みを保存しておく（図だけ描き直すときや，Mac に持ってきて見るとき用）
os.makedirs("../results", exist_ok=True)
np.savez(f"../results/umap_embeddings_{BACKEND}_n{N_SAMPLES}.npz",
         idx=idx, y=y_real, real=Z_real, **{f"k{k}": Z_moment[k] for k in KS})

In [ ]:
def scatter(ax, Z, y, title, xlim=None, ylim=None):
    for c, name in enumerate(cifar_classes):
        m = y == c
        ax.scatter(Z[m, 0], Z[m, 1], s=0.5, alpha=0.5, color=plt.cm.tab10(c), label=name, rasterized=True)
    ax.set_title(title)
    ax.set_xlabel("UMAP 1")
    ax.set_ylabel("UMAP 2")
    if xlim is not None:
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

# 全パネルで軸の範囲を揃える
Z_all = np.concatenate([Z_real] + [Z_moment[k] for k in KS])
pad = 0.5
xlim = (Z_all[:, 0].min() - pad, Z_all[:, 0].max() + pad)
ylim = (Z_all[:, 1].min() - pad, Z_all[:, 1].max() + pad)

fig, axes = plt.subplots(1, len(KS) + 1, figsize=(4 * (len(KS) + 1), 4.2), sharex=True, sharey=True)
scatter(axes[0], Z_real, y_real, "real (fit)", xlim, ylim)
for ax, k in zip(axes[1:], KS):
    scatter(ax, Z_moment[k], y_real, f"gram_k{k} (transform)", xlim, ylim)
axes[0].legend(markerscale=10, fontsize=7, loc="best")
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig(f"../figures/umap_real_fit_gram_k_transform_{BACKEND}_n{N_SAMPLES}.png", dpi=150)

## 埋め込みの変化量
- real での位置からの移動距離（同じ画像どうし）を k ごとに比較する

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
disp = [np.linalg.norm(Z_moment[k] - Z_real, axis=1) for k in KS]
ax.boxplot(disp, tick_labels=[f"k={k}" for k in KS], showfliers=False)
ax.set_ylabel("UMAP displacement from real")
for k, dd in zip(KS, disp):
    print(f"k={k}: mean={dd.mean():.3f}, median={np.median(dd):.3f}")
plt.tight_layout()
plt.savefig(f"../figures/umap_displacement_{BACKEND}_n{N_SAMPLES}.png", dpi=150)